In [24]:
pip install requests beautifulsoup4 pandas


[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
import os
print("当前工作目录:", os.getcwd())
print("文件是否存在:", os.path.exists("../data/raw/north_america/programmes.csv"))
print("文件大小:", os.path.getsize("../data/raw/north_america/programmes.csv"), "bytes")

当前工作目录: /Users/yuyuhang/Desktop/MGS4701/Group/mgs4701-track4-masters-landscape/notebooks
文件是否存在: True
文件大小: 2356 bytes


In [26]:
from pathlib import Path
Path("../data/clean").mkdir(parents=True, exist_ok=True)
print("创建成功")

创建成功


In [20]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1"
}

BASE = Path("../data/raw/north_america")
HTML_DIR = BASE / "html_2026"
TEXT_DIR = BASE / "text_2026"
HTML_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

robots_cache = {}

def check_robots(url):
    """检查 robots.txt 是否允许抓取"""
    parsed = urlparse(url)
    root = f"{parsed.scheme}://{parsed.netloc}"
    if root in robots_cache:
        return robots_cache[root]
    try:
        r = requests.get(root + "/robots.txt", headers=HEADERS, timeout=10)
        if r.status_code == 404:
            robots_cache[root] = True
            return True
        rp = RobotFileParser()
        rp.parse(r.text.splitlines())
        verdict = rp.can_fetch(HEADERS["User-Agent"], url)
        robots_cache[root] = verdict
        return verdict
    except Exception:
        robots_cache[root] = None
        return None

def fetch_and_save(url, programme_id, page_type):
    """下载一个页面，保存 HTML 和纯文本"""
    if not check_robots(url):
        return {"status": "robots_disallow", "url": url}

    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
    except Exception as e:
        return {"status": f"error: {type(e).__name__}", "url": url}

    r.encoding = r.apparent_encoding

    html_path = HTML_DIR / f"{programme_id}_{page_type}.html"
    text_path = TEXT_DIR / f"{programme_id}_{page_type}.txt"

    html_path.write_text(r.text, encoding="utf-8")

    soup = BeautifulSoup(r.text, "html.parser")
    text = soup.get_text(" ", strip=True)
    text_path.write_text(text, encoding="utf-8")

    return {
        "status": "ok",
        "url": url,
        "html_path": str(html_path),
        "text_path": str(text_path),
    }

# ============================================================
# 先只跑 P001，确认跑通
# ============================================================
TEST_PROGRAMME_ID = "P001"
TEST_URL = "https://mitsloan.mit.edu/master-of-business-analytics"

print(f"Testing: {TEST_PROGRAMME_ID} — {TEST_URL}")
result = fetch_and_save(TEST_URL, TEST_PROGRAMME_ID, "programme")

if result["status"] == "ok":
    print(f"  HTML saved: {result['html_path']}")
    print(f"  Text saved: {result['text_path']}")
    text_preview = Path(result["text_path"]).read_text(encoding="utf-8")[:300]
    print(f"\n  Text preview:\n  {text_preview}")
else:
    print(f"  Status: {result['status']}")

progs = pd.read_csv("../data/raw/north_america/programmes.csv")
rows = []

for _, row in progs.iterrows():
    pid = row["programme_id"]
    url = row["programme_url"]
    if pd.isna(url) or not str(url).startswith("http"):
        continue

    res = fetch_and_save(url, pid, "programme")
    rows.append({
        "programme_id": pid,
        "university": row["university"],
        "programme_name": row["programme_name"],
        "page_type": "programme",
        "url": url,
        "status": res["status"],
        "access_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    })
    time.sleep(1.5)

index_df = pd.DataFrame(rows)
index_df.to_csv("../data/clean/na_download_index.csv", index=False, encoding="utf-8-sig")
print(f"\nDone: {len(index_df)} records")
print(index_df[["programme_id", "status"]].to_string())

Testing: P001 — https://mitsloan.mit.edu/master-of-business-analytics
  HTML saved: ../data/raw/north_america/html_2026/P001_programme.html
  Text saved: ../data/raw/north_america/text_2026/P001_programme.txt

  Text preview:
  Explore the Program | Master of Business Analytics | MIT Sloan Skip to main content MIT Sloan logo academics Expert Insights Values Events Alumni Faculty About Executive Education academics contact facebook instagram linkedin twitter youtube MIT Dive deep into data science. Take the first step. Same

Done: 17 records
   programme_id           status
0          P001               ok
1          P002               ok
2          P004               ok
3          P005               ok
4          P006               ok
5          P007               ok
6          P008               ok
7          P009               ok
8          P010               ok
9          P013  robots_disallow
10         P014               ok
11         P015               ok
12         P016         

In [21]:
import requests
import pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

df = pd.read_csv("../data/raw/north_america/programmes.csv")
rows = []

for _, row in df.iterrows():
    pid = row["programme_id"]
    url = row["programme_url"]
    try:
        r = requests.get(url, headers=HEADERS, timeout=20, allow_redirects=True)
        status = r.status_code
        final = r.url
    except Exception as e:
        status = f"err:{type(e).__name__}"
        final = ""
    rows.append({
        "programme_id": pid,
        "url": url,
        "status": status,
        "final_url": final,
    })
    print(f"{pid}: {status}  {url}")

pd.DataFrame(rows).to_csv("../data/clean/na_url_check.csv", index=False, encoding="utf-8-sig")
print("\nSaved to ../data/clean/na_url_check.csv")

P001: 200  https://mitsloan.mit.edu/master-of-business-analytics
P002: 200  https://www.anderson.ucla.edu/degrees/master-of-science-in-business-analytics
P004: 200  https://www.fuqua.duke.edu/programs/mqm-business-analytics
P005: 200  https://www.johnson.cornell.edu/programs/master-of-science-in-business-analytics/
P006: 200  https://www.cmu.edu/tepper/programs/master-business-analytics/full-time/index.html
P007: 200  https://www.mccombs.utexas.edu/graduate/specialized-masters/ms-business-analytics-and-artificial-intelligence/on-campus/
P008: 200  https://www.marshall.usc.edu/programs/specialized-masters/master-science-business-analytics
P009: 200  https://foster.uw.edu/academics/degree-programs/master-science-business-analytics/
P010: 200  https://business.purdue.edu/masters/programs/ms-business-analytics-and-information-management/home.php
P013: 200  https://rotman.utoronto.ca/registrar/programs/specialized-programs/master-of-management-analytics/
P014: 200  https://www.mcgill.ca/des

In [22]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path
from datetime import datetime, timezone

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

BASE = Path("../data/raw/north_america")
HTML_DIR = BASE / "html_2026"
TEXT_DIR = BASE / "text_2026"
HTML_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)
Path("../data/clean").mkdir(parents=True, exist_ok=True)

def fetch_and_save(url, programme_id, page_type):
    """下载一个页面，保存 HTML 和纯文本"""
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
    except Exception as e:
        return {"status": f"error: {type(e).__name__}", "url": url}

    r.encoding = r.apparent_encoding

    html_path = HTML_DIR / f"{programme_id}_{page_type}.html"
    text_path = TEXT_DIR / f"{programme_id}_{page_type}.txt"

    html_path.write_text(r.text, encoding="utf-8")

    soup = BeautifulSoup(r.text, "html.parser")
    text = soup.get_text(" ", strip=True)
    text_path.write_text(text, encoding="utf-8")

    return {"status": "ok", "url": url, "html_path": str(html_path), "text_path": str(text_path)}


progs = pd.read_csv("../data/raw/north_america/programmes.csv")
rows = []

for _, row in progs.iterrows():
    pid = row["programme_id"]
    url = row["programme_url"]
    if pd.isna(url) or not str(url).startswith("http"):
        continue

    res = fetch_and_save(url, pid, "programme")
    rows.append({
        "programme_id": pid,
        "university": row["university"],
        "programme_name": row["programme_name"],
        "page_type": "programme",
        "url": url,
        "status": res["status"],
        "access_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    })
    print(f"{pid}: {res['status']}")
    time.sleep(1.5)

index_df = pd.DataFrame(rows)
index_df.to_csv("../data/clean/na_download_index.csv", index=False, encoding="utf-8-sig")
print(f"\nDone: {len(index_df)} records")
print(index_df[["programme_id", "status"]].to_string())

P001: ok
P002: ok
P004: ok
P005: ok
P006: ok
P007: ok
P008: ok
P009: ok
P010: ok
P013: ok
P014: ok
P015: ok
P016: ok
P017: ok
P018: ok
P019: ok
P020: ok

Done: 17 records
   programme_id status
0          P001     ok
1          P002     ok
2          P004     ok
3          P005     ok
4          P006     ok
5          P007     ok
6          P008     ok
7          P009     ok
8          P010     ok
9          P013     ok
10         P014     ok
11         P015     ok
12         P016     ok
13         P017     ok
14         P018     ok
15         P019     ok
16         P020     ok


In [23]:
from pathlib import Path

html_dir = Path("../data/raw/north_america/html_2026")
text_dir = Path("../data/raw/north_america/text_2026")

print("=== HTML 文件 ===")
for f in sorted(html_dir.glob("*.html")):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

print(f"\nHTML 文件总数: {len(list(html_dir.glob('*.html')))}")

print("\n=== TEXT 文件 ===")
for f in sorted(text_dir.glob("*.txt")):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

print(f"\nTEXT 文件总数: {len(list(text_dir.glob('*.txt')))}")

=== HTML 文件 ===
  P001_programme.html  (61857 bytes)
  P002_programme.html  (478071 bytes)
  P004_programme.html  (71234 bytes)
  P005_programme.html  (161409 bytes)
  P006_programme.html  (91138 bytes)
  P007_programme.html  (234262 bytes)
  P008_programme.html  (357938 bytes)
  P009_programme.html  (251911 bytes)
  P010_programme.html  (98698 bytes)
  P013_programme.html  (47387 bytes)
  P014_programme.html  (74652 bytes)
  P015_programme.html  (208627 bytes)
  P016_programme.html  (73380 bytes)
  P017_programme.html  (337801 bytes)
  P018_programme.html  (40980 bytes)
  P019_programme.html  (167621 bytes)
  P020_programme.html  (183577 bytes)

HTML 文件总数: 17

=== TEXT 文件 ===
  P001_programme.txt  (6752 bytes)
  P002_programme.txt  (33764 bytes)
  P004_programme.txt  (5540 bytes)
  P005_programme.txt  (9494 bytes)
  P006_programme.txt  (9217 bytes)
  P007_programme.txt  (8877 bytes)
  P008_programme.txt  (23221 bytes)
  P009_programme.txt  (6316 bytes)
  P010_programme.txt  (7092 byte

In [33]:
# 子链接抓取

from pathlib import Path
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

BASE = Path("../data/raw/north_america")
HTML_DIR = BASE / "html_2026"

progs = pd.read_csv(BASE / "programmes.csv")
base_urls = dict(zip(progs["programme_id"], progs["programme_url"]))

keywords = {
    "curriculum": ["curriculum", "course", "academic", "class"],
    "admission":  ["admission", "apply", "requirement", "prerequisite"],
    "tuition":    ["tuition", "fee", "cost", "financing"],
    "outcomes":   ["career", "outcome", "employment", "placement", "job"],
}

results = []

for pid, base_url in base_urls.items():
    html_file = HTML_DIR / f"{pid}_programme.html"
    if not html_file.exists():
        continue
    html = html_file.read_text(encoding="utf-8")
    soup = BeautifulSoup(html, "html.parser")

    found = {k: [] for k in keywords}
    for a in soup.select("a[href]"):
        href = a["href"]
        if href.startswith(("#", "mailto:", "tel:", "javascript:")):
            continue
        full = urljoin(base_url, href).split("#")[0].split("?")[0]
        # 只看同域链接
        if not full.startswith(base_url.split("/", 3)[0] + "//" + base_url.split("/")[2]):
            continue
        text = a.get_text(strip=True).lower()
        for page_type, words in keywords.items():
            if any(w in full.lower() or w in text for w in words):
                found[page_type].append(full)

    for page_type, urls in found.items():
        # 去重，取第一个
        uniq = list(dict.fromkeys(urls))
        results.append({
            "programme_id": pid,
            "page_type": page_type,
            "candidate_url": uniq[0] if uniq else "",
            "all_candidates": " | ".join(uniq[:5]),
        })

df = pd.DataFrame(results)
df.to_csv("../data/clean/na_subpage_candidates.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} candidate rows")
df.head(20)

Saved 68 candidate rows


,programme_id,page_type,candidate_url,all_candidates
0,P001,curriculum,https://mitsloan.mit.edu/master-of-business-an...,https://mitsloan.mit.edu/master-of-business-an...
1,P001,admission,https://mitsloan.mit.edu/mba/deferred-admission,https://mitsloan.mit.edu/mba/deferred-admissio...
2,P001,tuition,,
3,P001,outcomes,https://mitsloan.mit.edu/work-mit-sloan,https://mitsloan.mit.edu/work-mit-sloan
4,P002,curriculum,https://www.anderson.ucla.edu/specialized-acad...,https://www.anderson.ucla.edu/specialized-acad...
5,P002,admission,https://www.anderson.ucla.edu/apply,https://www.anderson.ucla.edu/apply | https://...
6,P002,tuition,https://www.anderson.ucla.edu/degrees/master-o...,https://www.anderson.ucla.edu/degrees/master-o...
7,P002,outcomes,https://www.anderson.ucla.edu/alumni/career-se...,https://www.anderson.ucla.edu/alumni/career-se...
8,P004,curriculum,https://www.fuqua.duke.edu/faculty-research/ac...,https://www.fuqua.duke.edu/faculty-research/ac...
9,P004,admission,https://www.fuqua.duke.edu/admissions,https://www.fuqua.duke.edu/admissions | https:...


In [34]:
import requests, time

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"}

df = pd.read_csv("../data/clean/na_subpage_candidates.csv")
df["status"] = ""

for i, row in df.iterrows():
    url = row["candidate_url"]
    if not url:
        df.at[i, "status"] = "empty"
        continue
    try:
        r = requests.get(url, headers=HEADERS, timeout=15, allow_redirects=True)
        df.at[i, "status"] = r.status_code
    except Exception as e:
        df.at[i, "status"] = f"err:{type(e).__name__}"
    time.sleep(1)

df.to_csv("../data/clean/na_subpage_candidates.csv", index=False, encoding="utf-8-sig")
print(df[["programme_id", "page_type", "status"]].to_string())

   programme_id   page_type             status
0          P001  curriculum                200
1          P001   admission                200
2          P001     tuition  err:MissingSchema
3          P001    outcomes                200
4          P002  curriculum                200
5          P002   admission                200
6          P002     tuition                200
7          P002    outcomes                200
8          P004  curriculum                200
9          P004   admission                200
10         P004     tuition                200
11         P004    outcomes                200
12         P005  curriculum  err:MissingSchema
13         P005   admission  err:MissingSchema
14         P005     tuition  err:MissingSchema
15         P005    outcomes  err:MissingSchema
16         P006  curriculum                200
17         P006   admission                200
18         P006     tuition  err:MissingSchema
19         P006    outcomes                200
20         P0

In [35]:
from datetime import datetime, timezone

df = pd.read_csv("../data/clean/na_subpage_candidates.csv")
df = df[df["status"] == 200]  # 只抓有效的

rows = []
for _, row in df.iterrows():
    pid = row["programme_id"]
    page_type = row["page_type"]
    url = row["candidate_url"]
    res = fetch_and_save(url, pid, page_type)
    rows.append({
        "programme_id": pid,
        "page_type": page_type,
        "url": url,
        "status": res["status"],
        "access_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    })
    print(f"{pid} {page_type}: {res['status']}")
    time.sleep(1.5)

pd.DataFrame(rows).to_csv("../data/clean/na_subpage_download_index.csv", index=False, encoding="utf-8-sig")

In [37]:
#终端 print 会被截断是因为输出太长。分两段打印：
import pandas as pd
df = pd.read_csv("../data/clean/na_subpage_candidates.csv")

# 前 40 行
print(df.iloc[:40][["programme_id", "page_type", "status", "candidate_url"]].to_string())

   programme_id   page_type             status                                                                                                                                              candidate_url
0          P001  curriculum                200                                                                                           https://mitsloan.mit.edu/master-of-business-analytics/meet-class
1          P001   admission                200                                                                                                            https://mitsloan.mit.edu/mba/deferred-admission
2          P001     tuition  err:MissingSchema                                                                                                                                                        NaN
3          P001    outcomes                200                                                                                                                    https://mitsloan.mit.edu/work-

In [38]:
# 后 40 行
print(df.iloc[40:][["programme_id", "page_type", "status", "candidate_url"]].to_string())

   programme_id   page_type             status                                                                                                     candidate_url
40         P014  curriculum                200                                                                https://www.mcgill.ca/desautels/academic-integrity
41         P014   admission  err:MissingSchema                                                                                                               NaN
42         P014     tuition  err:MissingSchema                                                                                                               NaN
43         P014    outcomes                200                                                                            https://www.mcgill.ca/desautels/career
44         P015  curriculum                200                         https://www.sauder.ubc.ca/programs/real-estate/real-estate-licensing-registration-courses
45         P015   admission       